# 🏗️ ResNet Implementation: Residual Connections on CIFAR-10

## Overview
This notebook implements a **mini ResNet** from scratch for CIFAR-10 image classification.  
We explore how **residual (skip) connections** solve the vanishing gradient problem and enable training of much deeper networks.

## 📖 What are Residual Connections?

### The Vanishing Gradient Problem
When training very deep networks, gradients become extremely small as they propagate backward through many layers — causing early layers to learn very slowly or not at all. This is the **vanishing gradient problem**.

### The ResNet Solution
ResNet (He et al., 2015) introduces **skip connections** (also called shortcut connections) that bypass one or more layers:

```
Output = F(x) + x
```

Where:
- `x` = the input (shortcut path)
- `F(x)` = the learned residual function (main path)
- `Output` = their element-wise sum

### Why it works:
- Gradients can flow directly through the shortcut path without shrinking
- The network only needs to learn the **residual** (difference), not a full transformation
- Enables training of networks with **100+ layers**

## 1️⃣ Import Libraries

In [ ]:
# Deep learning and visualization imports
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping
import numpy as np
import matplotlib.pyplot as plt

print("TensorFlow version:", tf.__version__)

# CIFAR-10 class labels
CLASS_NAMES = ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']

## 2️⃣ Load & Preprocess CIFAR-10

We use the original 32×32 resolution for our mini ResNet (no upscaling needed).

In [ ]:
# Load CIFAR-10 (no resizing needed for mini ResNet)
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()
y_train = y_train.ravel()
y_test = y_test.ravel()

# Normalize pixel values to [0, 1]
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

print(f"Training set:  {x_train.shape}")
print(f"Test set:      {x_test.shape}")

## 3️⃣ Build Residual Block

The residual block is the fundamental building block of ResNet.  
Each block contains two convolutional layers with a skip connection.

In [ ]:
def residual_block(x, filters, stride=1):
    """
    A standard residual block with a skip (shortcut) connection.
    
    Architecture:
        Conv2D -> BatchNorm -> ReLU -> Conv2D -> BatchNorm
        + skip connection (with optional 1x1 projection if shapes differ)
        -> ReLU
    """
    shortcut = x  # Save input for the skip connection

    # First convolutional layer
    x = layers.Conv2D(filters, kernel_size=3, strides=stride,
                      padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)

    # Second convolutional layer
    x = layers.Conv2D(filters, kernel_size=3, strides=1,
                      padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)

    # Projection shortcut: adjust dimensions if stride != 1 or channels differ
    # Use int_shape for reliable dimension comparison in both eager and graph mode
    if stride != 1 or tf.keras.backend.int_shape(shortcut)[-1] != filters:
        shortcut = layers.Conv2D(filters, kernel_size=1, strides=stride,
                                 padding='same', use_bias=False)(shortcut)
        shortcut = layers.BatchNormalization()(shortcut)

    # Add skip connection and apply final activation
    x = layers.Add()([x, shortcut])
    x = layers.Activation('relu')(x)
    return x

## 4️⃣ Build Mini ResNet for CIFAR-10

Our mini ResNet is designed for 32×32 CIFAR-10 images with 3 residual blocks.

In [ ]:
def build_mini_resnet(num_classes=10):
    """
    Build a mini ResNet tailored for CIFAR-10 (32x32 input).
    Architecture:
        Input -> Conv2D -> BN -> ReLU
        -> 3 Residual Blocks (16, 32, 64 filters)
        -> Global Average Pooling
        -> Dense(num_classes, softmax)
    """
    inputs = layers.Input(shape=(32, 32, 3), name='input')

    # Initial convolutional stem
    x = layers.Conv2D(16, kernel_size=3, strides=1, padding='same',
                      use_bias=False, name='stem_conv')(inputs)
    x = layers.BatchNormalization(name='stem_bn')(x)
    x = layers.Activation('relu', name='stem_relu')(x)

    # Residual blocks with increasing filter counts
    x = residual_block(x, filters=16)          # Block 1
    x = residual_block(x, filters=32, stride=2)  # Block 2 (spatial downsample)
    x = residual_block(x, filters=64, stride=2)  # Block 3 (spatial downsample)

    # Global average pooling replaces flatten + dense
    x = layers.GlobalAveragePooling2D(name='global_avg_pool')(x)

    # Classification head
    outputs = layers.Dense(num_classes, activation='softmax', name='output')(x)

    return models.Model(inputs, outputs, name='Mini_ResNet')


# Build and inspect the model
model = build_mini_resnet()
model.summary()

## 5️⃣ Compile & Train

In [ ]:
# Compile the mini ResNet
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# EarlyStopping to avoid overfitting
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True,
    verbose=1
)

# Train the model
print("Training Mini ResNet on CIFAR-10...")
history = model.fit(
    x_train, y_train,
    epochs=20,
    batch_size=128,
    validation_split=0.1,
    callbacks=[early_stop],
    verbose=1
)

## 6️⃣ Evaluate & Plot Results

In [ ]:
# Evaluate on test set
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
print(f"Test Accuracy : {test_acc:.4f}  ({test_acc*100:.2f}%)")
print(f"Test Loss     : {test_loss:.4f}")

# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy curves
axes[0].plot(history.history['accuracy'], label='Train Accuracy')
axes[0].plot(history.history['val_accuracy'], label='Val Accuracy')
axes[0].set_title('Mini ResNet — Accuracy', fontsize=13)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Loss curves
axes[1].plot(history.history['loss'], label='Train Loss')
axes[1].plot(history.history['val_loss'], label='Val Loss')
axes[1].set_title('Mini ResNet — Loss', fontsize=13)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 🔑 Key Takeaways

| Concept | Detail |
|---|---|
| **Residual connections** | Allow gradients to flow directly, solving vanishing gradient |
| **Skip connections** | The network learns residuals F(x) = H(x) - x |
| **BatchNormalization** | Used after every Conv2D for training stability |
| **Global Average Pooling** | Reduces parameters vs. Flatten + Dense |
| **Mini ResNet** | Achieves ~75–80% on CIFAR-10 with just 3 residual blocks |

### Conclusion
Residual connections are one of the most impactful innovations in deep learning history.  
Our mini ResNet demonstrates how even a small residual network effectively classifies CIFAR-10, and the same principles scale to ResNet-50, ResNet-101, and beyond.